# Module 2.3 — Iterators & Generators

### Python Mastery · Track 2: Data Structures & Iteration

Every time you write a `for` loop, an iteration protocol works behind the scenes. Understanding it lets you create your own iterable objects and write generators, which produce sequences lazily and elegantly. This module covers the protocol, generator functions, `yield from`, and custom iterable classes.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- Step through the protocol manually with `next()` to see how a loop actually works.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Explain the iteration protocol and the roles of `iter()` and `next()`.
2. Distinguish an iterable from an iterator.
3. Write generator functions with `yield` and reason about lazy evaluation.
4. Delegate to another iterable with `yield from`.
5. Build a custom iterable class with `__iter__` and `__next__`.

**Prerequisites:** Track 1 and Modules 2.1 to 2.2.

---

## Part 1 · The Iteration Protocol

A `for` loop does not have built-in knowledge of lists or strings. Instead it relies on a protocol:

1. It calls `iter(object)` to obtain an **iterator**.
2. It calls `next(iterator)` repeatedly to get each value.
3. When there are no more values, `next` raises `StopIteration`, and the loop ends.

An **iterable** is anything you can call `iter()` on (lists, strings, dictionaries, files). An **iterator** is the object that actually produces values one at a time via `next()`. The cell below performs by hand what a `for` loop does automatically.

In [ ]:
colors = ["red", "green", "blue"]

it = iter(colors)            # obtain an iterator from the iterable
print(next(it))              # red
print(next(it))              # green
print(next(it))              # blue

# A fourth call would raise StopIteration; we catch it to show the end.
try:
    next(it)
except StopIteration:
    print("iterator exhausted (StopIteration raised)")

An important consequence: an iterator is consumed as you go and cannot be rewound. A list is an iterable you can loop over many times, but the iterator you get from it is single-use.

In [ ]:
numbers = [1, 2, 3]

it = iter(numbers)
print("first pass via iterator:", list(it))    # [1, 2, 3]
print("second pass via same iterator:", list(it))  # [] because it is exhausted

# The original list is unaffected and can be iterated again.
print("the list itself, again:", list(numbers))

## Part 2 · Generator Functions

A generator function looks like an ordinary function but uses `yield` instead of (or alongside) `return`. Calling it does not run the body; it returns a **generator object**, which is an iterator. Each time `next()` is called, the function runs until the next `yield`, hands back that value, and **pauses**, remembering all its local state. It resumes from there on the following call.

This makes generators ideal for producing sequences without storing them all at once.

In [ ]:
def count_up_to(limit):
    """Yield the numbers 1 through limit, one at a time."""
    n = 1
    while n <= limit:
        yield n          # hand back n and pause here until the next request
        n += 1

gen = count_up_to(3)     # nothing has run yet; we have a generator object
print("object:", gen)
print(next(gen))         # runs until the first yield -> 1
print(next(gen))         # resumes, runs to the next yield -> 2
print("rest via loop:", list(gen))   # the loop consumes the remaining values -> [3]

Because state is preserved between yields, generators express sequences that would be awkward otherwise, including ones that are effectively infinite. You take only as many values as you need.

In [ ]:
def fibonacci():
    """Yield Fibonacci numbers indefinitely."""
    a, b = 0, 1
    while True:           # an infinite generator; safe because the caller stops it
        yield a
        a, b = b, a + b

fib = fibonacci()
first_ten = [next(fib) for _ in range(10)]
print("first ten Fibonacci numbers:", first_ten)

## Part 3 · `yield from` — Delegating to Another Iterable

When a generator wants to yield every value from another iterable, `yield from` does so directly, without an explicit loop. It is clearer and also forwards values correctly when generators are chained.

In [ ]:
def chain_iterables(*iterables):
    """Yield all values from each iterable in turn."""
    for it in iterables:
        yield from it          # equivalent to: for value in it: yield value

combined = chain_iterables([1, 2], (3, 4), "ab")
print(list(combined))          # [1, 2, 3, 4, 'a', 'b']

# yield from also composes generators cleanly.
def countdown(n):
    while n > 0:
        yield n
        n -= 1

def launch():
    yield from countdown(3)
    yield "lift off"

print(list(launch()))

## Part 4 · Custom Iterable Classes

You can make your own objects iterable by implementing the protocol. There are two common patterns:

- Implement `__iter__` as a generator (simplest and recommended in most cases).
- Implement `__iter__` to return an iterator object that has both `__iter__` and `__next__`.

The first cell shows the simple generator-based approach; the second shows the fully explicit form so you understand what the protocol requires.

In [ ]:
class Countdown:
    """An iterable that counts down from a starting number to 1."""
    def __init__(self, start):
        self.start = start

    def __iter__(self):
        # Implementing __iter__ as a generator is concise and correct.
        n = self.start
        while n > 0:
            yield n
            n -= 1

for value in Countdown(4):
    print(value, end=" ")
print()
print("as a list:", list(Countdown(4)))   # works in any iteration context

In [ ]:
class Squares:
    """The fully explicit form: a separate iterator with __iter__ and __next__."""
    def __init__(self, limit):
        self.limit = limit

    def __iter__(self):
        self.current = 1
        return self                  # the object is its own iterator here

    def __next__(self):
        if self.current > self.limit:
            raise StopIteration      # signal the end exactly as the protocol requires
        value = self.current ** 2
        self.current += 1
        return value

print("squares:", list(Squares(5)))

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — Stepping through an iterator (Easy)

In [ ]:
letters = iter("abc")
print(next(letters))    # a
print(next(letters))    # b
print(next(letters))    # c
# Experiment: add a fourth next() inside a try/except to see StopIteration.

### Example 2 — A simple generator (Easy)

In [ ]:
def even_numbers(limit):
    """Yield even numbers from 0 up to but not including limit."""
    n = 0
    while n < limit:
        yield n
        n += 2

print(list(even_numbers(10)))
# Experiment: change the step to 3 to yield multiples of three.

### Example 3 — A generator that reads "records" lazily (Medium)

In [ ]:
def read_records(lines):
    """Yield non-empty, stripped lines as records, skipping blanks."""
    for line in lines:
        cleaned = line.strip()
        if cleaned:                 # skip empty lines
            yield cleaned

raw = ["  alpha ", "", "beta", "   ", "gamma  "]
for record in read_records(raw):
    print("record:", record)

### Example 4 — Pairing consecutive items with a generator (Medium)

In [ ]:
def pairwise(values):
    """Yield consecutive overlapping pairs: (v0,v1), (v1,v2), ..."""
    previous = None
    first = True
    for value in values:
        if not first:
            yield (previous, value)
        previous = value
        first = False

print(list(pairwise([10, 20, 30, 40])))    # [(10,20),(20,30),(30,40)]

### Example 5 — Composing generators with `yield from` (Difficult)

In [ ]:
def numbers_up_to(n):
    yield from range(1, n + 1)

def repeated(value, times):
    yield from [value] * times

def playlist():
    """Build a sequence by delegating to several generators."""
    yield from numbers_up_to(3)
    yield from repeated("-", 2)
    yield from numbers_up_to(2)

print(list(playlist()))    # [1, 2, 3, '-', '-', 1, 2]

### Example 6 — A custom iterable with state (Difficult)

In [ ]:
class Cyclic:
    """Iterate over a list, repeating it, for a fixed number of total items."""
    def __init__(self, items, total):
        self.items = items
        self.total = total

    def __iter__(self):
        count = 0
        index = 0
        while count < self.total:
            yield self.items[index % len(self.items)]   # wrap around with modulo
            index += 1
            count += 1

print(list(Cyclic(["a", "b", "c"], 7)))   # a b c a b c a
# Experiment: change total or the items list.

---

## Exercises

**Exercise 1 (Easy).** Use `iter()` and `next()` to print the first two items of the list `["x", "y", "z"]` by hand, without a loop.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Write a generator function `countdown(n)` that yields the numbers from `n` down to 1. Print the result as a list for `n = 5`.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Write a generator `squares_up_to(limit)` that yields the squares 1, 4, 9, ... while the square is less than or equal to `limit`. Show its output for `limit = 50`.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Write a generator `flatten(nested)` that, given a list of lists, yields every inner value in order, using `yield from`. Test it on `[[1, 2], [3], [4, 5, 6]]`.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Write a custom iterable class `Range2D(rows, cols)` whose iteration yields every `(row, col)` coordinate, row by row, for a grid of the given size. Print all coordinates for a 2 by 3 grid.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
it = iter(["x", "y", "z"])
print(next(it))
print(next(it))

**Solution 2**

In [ ]:
def countdown(n):
    while n > 0:
        yield n
        n -= 1

print(list(countdown(5)))

**Solution 3**

In [ ]:
def squares_up_to(limit):
    n = 1
    while n * n <= limit:
        yield n * n
        n += 1

print(list(squares_up_to(50)))

**Solution 4**

In [ ]:
def flatten(nested):
    for sublist in nested:
        yield from sublist

print(list(flatten([[1, 2], [3], [4, 5, 6]])))

**Solution 5**

In [ ]:
class Range2D:
    def __init__(self, rows, cols):
        self.rows = rows
        self.cols = cols

    def __iter__(self):
        for r in range(self.rows):
            for c in range(self.cols):
                yield (r, c)

print(list(Range2D(2, 3)))

---

## Summary and Key Points

- The iteration protocol underlies every `for` loop: `iter()` produces an iterator, `next()` yields each value, and `StopIteration` signals the end.
- An iterable can be iterated many times; an iterator is single-use and is consumed as you go.
- A generator function uses `yield`, returns a generator object, and pauses between values while preserving local state, which suits large or infinite sequences.
- `yield from` delegates to another iterable, simplifying chaining and composition of generators.
- Custom iterables implement `__iter__` (most simply as a generator); the fully explicit form provides `__iter__` and `__next__` and raises `StopIteration` at the end.

### Next module: 2.4 — Advanced Collections

The next module explores the `collections` module: `namedtuple`, `deque`, `defaultdict`, `Counter`, `OrderedDict`, and `ChainMap`.